# Corrected Stats SA Aggregation — Retail (P6242.1) + Electricity (P4141)

Rebuilds the modelling dataset (1986–2026) with two corrected Stats SA series:

- **Retail target:** Total retail trade sales, **constant 2019 prices, NSA**, growth-spliced across the 1995→2019 base change.
- **Electricity:** national production **Index (base 2019=100, NSA)** — the correct single series, replacing the previous blended average.

Both fix the same bug: the old pipeline dropped the H-label columns and averaged every category row. Validated against the official P6242.1 May 2026 release.

In [5]:
"""
build_retail_dataset_corrected.py
=================================
Corrected aggregation of the Stats SA P6242.1 retail trade sales target and
assembly of the monthly modelling dataset (1986-2026).

WHY THIS SCRIPT EXISTS
----------------------
The previous aggregation averaged *every* category row per date because the
category-label columns (H01-H05) were dropped by a `startswith('h')` filter,
which left no id columns and silently skipped the "Total" selection. That
averaged constant + current prices, seasonally adjusted + not, and totals +
sub-categories together, and because each source file has a different row
composition the blended level jumped at file boundaries. That is what produced
the spurious 2004 block (a ~10x, one-year artifact).

THE FIX (three principles)
--------------------------
1. Do NOT drop the H-columns. Use them to select ONE consistent published
   series: the TOTAL, at CONSTANT prices, NOT seasonally adjusted (NSA).
2. Base years differ across vintages (pre-2002 files are constant 1995 prices;
   the 2002+ file is constant 2019 prices). Splice by GROWTH RATES (chain the
   pre-2002 history onto the modern 2019-base anchor), which removes the base
   seam while preserving each era's real dynamics.
3. Recompute every derived feature (lags, growth) from the corrected target so
   nothing inherits the old blended values.

VALIDATION
----------
The output matches the official P6242.1 (May 2026) release exactly:
Dec 2025 = 138,834; Jan-May 2026 = 97,326 / 95,209 / 99,422 / 96,677 / 101,008
(constant 2019 prices, R million).

NOTES / CAVEATS
---------------
* Target is constant 2019 prices, NSA (keeps seasonality for the models).
  Switch PRICE_BASIS to 'cur' for the current-price (nominal) total.
* The pre-2002 total is on the old "type of merchandise" classification and the
  2002+ total on "type of dealer" (a Stats SA survey redesign). Their December
  seasonal amplitude differs mildly (Dec/annual-mean ~1.47 pre-2002 vs ~1.38
  from 2002); disclose this in the data section. Restricting to 2002-2026 avoids
  it entirely if a single-definition series is preferred.
* Covariates (CPI, CCI, exchange rate) come from single-series exports and are
  attached as-is. `electricity` was aggregated with the same buggy averaging in
  the original pipeline; it is the noise variable dropped from the models, so it
  is carried through here with a warning rather than re-derived. Set
  FIX_ELECTRICITY=True to select its Total row analogously if needed.
* Exogenous covariates are only available to Dec 2025; 2026 cells are frozen
  (forward-filled) and must NOT be treated as real 2026 information. The
  Jan-May 2026 holdout is target-only and forecast from the Dec 2025 origin.
* Consumer confidence (CCI) is published QUARTERLY while the model is monthly.
  It is disaggregated with a step function (each reading carried forward until
  the next), not interpolated. See disaggregate_quarterly_to_monthly() for why:
  interpolation and regression-based disaggregation both use readings that are
  not yet observable at the month being filled, which would break the leakage
  control the rest of the pipeline enforces.
"""

import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
PRICE_BASIS = "con"      # 'con' = constant prices ; 'cur' = current prices
START = "1986-01-01"
END = "2026-05-31"
FIX_ELECTRICITY = True   # True: extract the correct national production index from P4141 files
                         # False: carry the (blended) electricity column from COVARIATE_SOURCE

# Electricity: P4141 "Physical volume of electricity production", Index, base 2019=100,
# NOT seasonally adjusted (H03 == 'ELEKIN11'). Already on a single 2019 base across all
# files, so no splicing is needed. NOTE: the old "CV > 31, drop for noise" criterion was
# an artifact of the blend and of applying CV to a near-zero-mean (flat) growth series;
# the corrected index is smooth. Whether to keep it should rest on the ablation, not CV.
ELECTRICITY_FILES = [
    "Excel table from 1985 to 1989_Electricity.xlsx",
    "Excel table from 1990 to 1999_Electricity.xlsx",
    "Excel table from 2000_Electricity.xlsx",
]
ELECTRICITY_CODE = "ELEKIN11"  # production index, base 2019=100, NSA

RETAIL_FILES = {
    # name: (path, engine)
    "1968_1980": ("Excel 1968 to 1980.xls", "xlrd"),
    "1981_1990": ("Excel 1981 to 1990.xls", "xlrd"),
    "1991_2000": ("Excel 1991 to 2000.xls", "xlrd"),
    "from_2001": ("Excel from 2001.xls", "xlrd"),
    "from_2002": ("Retail trade sales from 2002.xlsx", None),
}
# Path to the previous dataset that carries the real covariate columns.
COVARIATE_SOURCE = "final_model_dataset_RS_AugmentedUntil202605.csv"

# CCI is a QUARTERLY survey; these four columns hold a value in the reading
# months and NaN elsewhere. They are disaggregated to monthly by step function.
CCI_COLS = [
    "cci_net_balance",
    "cci_expected_economic_performance",
    "cci_expected_household_finances",
    "cci_time2buy_durables_net",
]

# Months between the end of a CCI reference quarter and its BER publication
# date. 0 treats a reading as available from the final month of its own
# quarter, which reproduces the original behaviour. Set to the true delay for a
# real-time treatment.
#
# VERIFY AGAINST BER RELEASE DATES BEFORE CHANGING. Raising this to 1 shifts
# every cci_level observation by a month, which changes the tree-based feature
# matrix and therefore the walk-forward results. SARIMAX is unaffected, since
# cci_level fails the exogenous screen on its coefficient of variation either
# way. Changing it requires re-running the EDA and modelling notebooks.
CCI_PUBLICATION_LAG_MONTHS = 0


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def _mo_to_date(col):
    """'mo011986' or 'MO011986' -> Timestamp('1986-01-01')."""
    c = str(col).upper().replace("MO", "")
    return pd.Timestamp(f"{c[2:]}-{c[:2]}-01")


def _read(path, engine):
    df = pd.read_excel(path, engine=engine) if engine else pd.read_excel(path)
    df.columns = [str(c) for c in df.columns]
    return df


def historical_total(path, engine):
    """
    Select the TOTAL, constant-price, ACTUAL (NSA) row from a pre-2002 file
    (old classification; H15 holds the price basis, H16 the adjustment,
    H05 == 'TOTAL' marks the total row).
    """
    df = _read(path, engine)
    mo = [c for c in df.columns if str(c).upper().startswith("MO")]
    up = lambda n: df[n].astype(str).str.upper()
    mask = (
        up("H15").str.contains("CONSTANT", na=False)
        & up("H16").str.contains("ACTUAL", na=False)
        & up("H05").str.fullmatch("TOTAL", na=False)
    )
    sel = df[mask]
    if len(sel) == 0:
        raise ValueError(f"No constant/actual/total row found in {path}")
    s = sel[mo].iloc[0].astype(float)
    return pd.Series(s.values, index=[_mo_to_date(c) for c in mo]).sort_index()


def modern_total(path, basis="con"):
    """
    Select the modern (2002+) Total row. H03 codes:
      con_act  = constant prices, actual (NSA)   <-- default
      con_seas = constant prices, seasonally adjusted
      cur_act  = current  prices, actual (NSA)
      cur_seas = current  prices, seasonally adjusted
    """
    code = f"{basis}_act"  # NSA total
    df = _read(path, None)
    mo = [c for c in df.columns if str(c).upper().startswith("MO")]
    sel = df[df["H03"] == code]
    if len(sel) == 0:
        raise ValueError(f"No H03=={code} Total row in {path}")
    s = sel[mo].iloc[0].astype(float)
    return pd.Series(s.values, index=[_mo_to_date(c) for c in mo]).sort_index()


def electricity_index(files=ELECTRICITY_FILES, code=ELECTRICITY_CODE):
    """
    Select the national electricity production index (Index, base 2019=100, NSA)
    from the P4141 files by its H03 code, and concatenate across files. Already
    on one 2019 base, so a direct splice suffices (no growth-chaining needed).
    """
    parts = []
    for f in files:
        df = _read(f, None)
        mo = [c for c in df.columns if str(c).upper().startswith("MO")]
        row = df[df["H03"] == code]
        if len(row) == 0:
            raise ValueError(f"No H03=={code} row in {f}")
        s = row[mo].iloc[0].astype(float)
        parts.append(pd.Series(s.values, index=[_mo_to_date(c) for c in mo]))
    s = pd.concat(parts).sort_index()
    s = s[~s.index.duplicated(keep="last")]
    s.name = "electricity"
    return s


def growth_splice(hist, modern, join="2002-01-01"):
    """
    Keep `modern` from the join date onward; back-cast the pre-join history by
    applying the historical series' own month-on-month growth rates to the
    modern anchor. Removes the base-year seam by construction.
    """
    join = pd.Timestamp(join)
    out = {d: v for d, v in modern.loc[join:].items()}
    hist_g = hist.pct_change()
    back_dates = pd.date_range(START, join - pd.offsets.MonthBegin(1), freq="MS")
    prev = join
    for d in reversed(back_dates):
        g = hist_g.get(prev, np.nan)
        out[d] = out[prev] if (pd.isna(g) or g == -1) else out[prev] / (1.0 + g)
        prev = d
    s = pd.Series(out).sort_index()
    return s[~s.index.duplicated(keep="last")]


# --------------------------------------------------------------------------- #
# 1. Build the corrected TARGET
# --------------------------------------------------------------------------- #
def build_target(basis=PRICE_BASIS):
    h81 = historical_total(*RETAIL_FILES["1981_1990"])
    h91 = historical_total(*RETAIL_FILES["1991_2000"])
    h01 = historical_total(*RETAIL_FILES["from_2001"])
    # Same 1995 base across the three historical files -> direct splice.
    hist = pd.concat(
        [h81.loc[:"1990-12"], h91.loc["1991-01":"2000-12"], h01.loc["2001-01":]]
    ).sort_index()
    hist = hist[~hist.index.duplicated(keep="last")]

    modern = modern_total(RETAIL_FILES["from_2002"][0], basis=basis)  # constant 2019
    target = growth_splice(hist, modern, join="2002-01-01").loc[START:END]
    target.name = "retail_sales"
    return target


# --------------------------------------------------------------------------- #
# 1b. Quarterly -> monthly disaggregation of the CCI block
# --------------------------------------------------------------------------- #
def disaggregate_quarterly_to_monthly(cov, cci_cols, lag_months=0):
    """Spread quarterly CCI readings across monthly observations (step function).

    Each quarterly reading is carried forward to the months that follow it until
    the next reading arrives. The alternatives were considered and rejected:

      * Linear interpolation between quarterly points makes the value assigned
        to the first month of a quarter depend on the reading at the END of
        that quarter, which is not observable at the time. That is future
        information entering a predictor.
      * Spline smoothing has the same defect over a wider span of quarters.
      * Regression-based temporal disaggregation (Chow-Lin 1971; Fernandez;
        Litterman) is the standard econometric treatment, but it is fitted on
        the full quarterly series and carries the same leakage unless it is
        re-estimated at every one of the 307 forecast origins. Its summation
        constraint also suits FLOW variables rather than a point-in-time
        sentiment index.

    The step function is the only one of the four that uses nothing beyond the
    most recently published reading, which is what a forecaster would have had.
    It introduces discontinuities at quarter boundaries; that is a real cost,
    accepted in exchange for a causally valid predictor.

    Months before the first available reading are left NaN rather than
    back-filled, so no value is manufactured from a later observation.
    """
    out = cov.copy()

    present = [c for c in cci_cols if c in out.columns]
    if not present:
        return out

    observed = out[present].dropna(how="all")
    if observed.empty:
        return out

    if lag_months:
        observed = observed.copy()
        observed.index = observed.index + pd.offsets.MonthEnd(lag_months)

    carried = observed.reindex(out.index.union(observed.index)).ffill()
    out[present] = carried.reindex(out.index)
    return out


def audit_cci(df, cci_cols=None, n=18):
    """Sanity check: CCI should be a step function changing four times a year."""
    cci_cols = cci_cols or CCI_COLS
    cols = [c for c in cci_cols if c in df.columns] + ["cci_level"]
    print("CCI tail (step function expected, one change per quarter):")
    print(df[cols].tail(n).to_string())
    changed = df["cci_level"].diff().ne(0) & df["cci_level"].notna()
    print("\nChanges in cci_level per year (4 expected once readings begin):")
    print(changed.groupby(df.index.year).sum().tail(12).to_string())


# --------------------------------------------------------------------------- #
# 2. Assemble the modelling dataset (target + covariates + derived features)
# --------------------------------------------------------------------------- #
def build_dataset():
    target = build_target()
    target.index = target.index + pd.offsets.MonthEnd(0)  # month-end to match covariates

    cov_cols = [
        "cpi", "electricity", "cci_net_balance", "cci_expected_economic_performance",
        "cci_expected_household_finances", "cci_time2buy_durables_net", "exchangeRate",
    ]
    old = pd.read_csv(COVARIATE_SOURCE, sep=";")
    old["date"] = pd.to_datetime(old["date"])
    cov = old.set_index("date")[cov_cols].sort_index()

    df = pd.DataFrame(index=target.index)
    df["retail_sales"] = target
    df = df.join(cov, how="left")

    # Replace the blended electricity column with the corrected national production
    # index (P4141, base 2019=100, NSA), aligned to month-end dates.
    if FIX_ELECTRICITY:
        elec = electricity_index()
        elec.index = elec.index + pd.offsets.MonthEnd(0)
        df["electricity"] = elec.reindex(df.index)

    # (1) CCI is quarterly and the model is monthly. Carry each reading forward
    # across the months it covers. See disaggregate_quarterly_to_monthly() for
    # why a step function is used rather than interpolation or Chow-Lin.
    df = disaggregate_quarterly_to_monthly(df, CCI_COLS, CCI_PUBLICATION_LAG_MONTHS)

    # (2) Covariates end late 2025 / early 2026; forward-fill so columns are
    # non-empty over the holdout. These frozen cells are NOT real 2026
    # information (the holdout is target-only, forecast from the Dec 2025
    # origin). This is a separate operation from (1) and is justified
    # differently: (1) is a frequency conversion, (2) is end-of-sample padding.
    df[cov_cols] = df[cov_cols].ffill()

    # Derived features recomputed from the CORRECTED target.
    df["retail_lag_1"] = df["retail_sales"].shift(1)
    df["retail_lag_3"] = df["retail_sales"].shift(3)
    df["retail_lag_6"] = df["retail_sales"].shift(6)
    df["retail_growth"] = df["retail_sales"].pct_change()
    df["electricity_growth"] = df["electricity"].pct_change()
    df["cci_level"] = df[
        ["cci_net_balance", "cci_expected_economic_performance",
         "cci_expected_household_finances", "cci_time2buy_durables_net"]
    ].mean(axis=1)

    schema = [
        "retail_sales", "cpi", "electricity", "cci_net_balance",
        "cci_expected_economic_performance", "cci_expected_household_finances",
        "cci_time2buy_durables_net", "exchangeRate", "retail_lag_1", "retail_lag_3",
        "retail_lag_6", "retail_growth", "electricity_growth", "cci_level",
    ]
    df = df[schema]
    df.index.name = "date"
    return df


# --------------------------------------------------------------------------- #
# 3. Validate against the official release and save
# --------------------------------------------------------------------------- #
def validate(df):
    checks = {
        "2025-12-31": 138834, "2026-01-31": 97326, "2026-02-28": 95209,
        "2026-03-31": 99422, "2026-04-30": 96677, "2026-05-31": 101008,
    }
    ok = True
    for d, v in checks.items():
        got = float(df.loc[d, "retail_sales"])
        flag = "OK" if abs(got - v) < 1 else "MISMATCH"
        ok = ok and flag == "OK"
        print(f"  {d}: ours={got:,.0f}  release={v:,d}  [{flag}]")
    print(f"  -> release validation {'PASSED' if ok else 'FAILED'}")
    return ok

## Build, validate and save

In [6]:
data = build_dataset()
print(data.shape)
validate(data)
out = data.reset_index()
out['date']=out['date'].dt.strftime('%Y/%m/%d')
out.to_csv('final_model_dataset_corrected.csv', sep=';', index=False)
data[['retail_sales','electricity']].describe()

# CCI frequency-conversion check (step function, 4 changes per year)
audit_cci(data)


WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
(485, 14)
  2025-12-31: ours=138,834  release=138,834  [OK]
  2026-01-31: ours=97,326  release=97,326  [OK]
  2026-02-28: ours=95,209  release=95,209  [OK]
  2026-03-31: ours=99,422  release=99,422  [OK]
  2026-04-30: ours=96,677  release=96,677  [OK]
  2026-05-31: ours=101,008  release=101,008  [OK]
  -> release validation PASSED
CCI tail (step function expected, one change per quarter):
            cci_net_balance  cci_expected_economic_performance  cci_expected_household_finances  cci_time2buy_durables_net  cci_level
date                                                                                                                                 
2024-12-31        -6.220493                          -8.803612                        10.875332                 -20.73319